# Video Anomaly Detection on UCF-Crime

This notebook reproduces the main results from our two-stream Multiple Instance Learning (MIL) approach for weakly-supervised video anomaly detection on the UCF-Crime dataset.

## Architecture Overview

Our pipeline consists of:
1. **Two-Stream Feature Extraction**: R3D-18 backbones (pre-trained on Kinetics-400) extract 512-D features from RGB frames and Sobel motion gradients
2. **MIL Aggregation**: Temporal attention networks process video-level bags of features with ranking loss
3. **Late Fusion**: Weighted combination of RGB and motion anomaly scores

## Setup

This project uses `uv` for dependency management. To install:

```bash
uv sync
```

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.ndimage import gaussian_filter1d

# Project imports
from src.models.mil import MILModel

# Set plotting style
sns.set_style("whitegrid")
sns.set_palette("muted")

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Configuration

Paths to pre-trained checkpoints and pre-extracted features.

In [ ]:
# CHECKPOINTS
RGB_MIL_CHECKPOINT = (
    "/work3/s225224/ucf-crime/checkpoints/mil/mil_rgb_20251204_130129/best_model.pth"
)
MOTION_MIL_CHECKPOINT = (
    "/work3/s225224/ucf-crime/checkpoints/mil/mil_motion_20251204_130253/best_model.pth"
)

# FEATURES (Pre-extracted from R3D backbones)
RGB_FEATURES_DIR = Path("/work3/s225224/ucf-crime/features/rgb_old2/Test")
MOTION_FEATURES_DIR = Path("/work3/s225224/ucf-crime/features/motion/Test")

# GROUND TRUTH
ANNOTATION_FILE = (
    "/work3/s225224/ucf-crime/data/Temporal_Anomaly_Annotation_for_Testing_Videos.txt"
)

# SETTINGS
INPUT_DIM = 512
STRIDE = 16
